# Scoring a graph against a hand-checked reference

`measure_kg.ipynb` measures any contract without labels, and pays for it: its triggers are
a floor, not an estimate. A paragraph that says *"represents and warrants that"* raises no
trigger, so a whole section of warranties can vanish without moving a number.

This notebook trades breadth for truth. For each contract that has a reference in
`infra/json/gold/`, it says which statements the graph should hold and scores every logged
extraction of that contract against it. The reference is written once, by reading the whole
contract, and each of its spans is checked to be verbatim.

It is meant for comparing extraction runs of the **same** contract — models, prompts,
designs — never for ranking contracts against each other.

In [1]:
import hashlib
import json
import re
import unicodedata
from pathlib import Path

ROOT = Path.cwd().parents[1] if Path.cwd().name == "KG" else Path.cwd()
GOLD_DIR = ROOT / "infra/json/gold"
RUNS_DIR = ROOT / "infra/json/kg_runs"
KG_DIR = ROOT / "infra/json/kg"
PARAGRAPHS_DIR = ROOT / "infra/json/paragraphs"

print(ROOT, "|", len(list(GOLD_DIR.glob("*.json"))), "reference(s)")

/home/sante/Documents/FGV/me/Contract-Knowledge-Graph | 1 reference(s)


## Matching

A gold statement and a graph statement are the same statement when they share a paragraph
and their texts overlap: one is mostly contained in the other, once the words every
sentence of a contract shares are set aside. Assignment is one-to-one, best overlap first;
agreeing on kind and parties breaks ties, which is what pulls apart the two statements of a
bilateral provision — same text, different burdened party.

What each gold statement counts for:

| flag | counted as |
|---|---|
| *(none)* | **core**: a miss lowers recall |
| `optional` | defensible either way: found or not, recall does not move, and its kind and parties are not scored |
| `nonParty` | the arbitrators' duties: neither a hit nor a miss |
| `burdenAlso` | other burdens that also count as right — where the prompt itself asks for one |

A graph statement with no gold counterpart is **non-deontic** when it was built from a span
the reference lists as stating no duty (governing law, deeming rules, recitals), a
**duplicate** when it repeats a statement already matched, and **spurious** otherwise.
Precision counts all three against the graph.

The expected clause of a statement is the clause of its paragraph in the outline of the
paragraph file, which for these contracts was checked against the text by reading.

In [2]:
STATEMENT_COLLECTIONS = {"obligations": "obligation", "rights": "right", "prohibitions": "prohibition"}

# Words every sentence of this corpus shares: left in, two unrelated sentences of one
# paragraph overlap enough to look like the same statement.
STOPWORDS = set(
    "a an the of and or to in on at by for from with as is are be been shall will may must "
    "not no any all such this that its their each other either both party parties agreement "
    "hereunder herein under".split()
)
WORD = re.compile(r"[a-z0-9]+")


def norm(text):
    text = unicodedata.normalize("NFKC", text or "").replace("’", "'").replace("“", '"').replace("”", '"')
    return " ".join(text.lower().split())


def tokens(text):
    return {w for w in WORD.findall(norm(text)) if w not in STOPWORDS}


def short(pid):
    """'root::...-p-13' and 'p-13' both become 'p-13'."""
    return "p-" + re.search(r"p-(\d+)$", str(pid)).group(1)


def containment(a, b):
    """(share of a found in b, share of b found in a)."""
    ta, tb = tokens(a), tokens(b)
    if not ta or not tb:
        return 0.0, 0.0
    inter = len(ta & tb)
    return inter / len(ta), inter / len(tb)


def similar(gold_text, kg_text):
    cg, ck = containment(gold_text, kg_text)
    f1 = 0.0 if cg + ck == 0 else 2 * cg * ck / (cg + ck)
    # A KG text that copies the whole gold span (and more) still points at it, and so does
    # one that copies only part of it — as long as that part says something.
    part_of_gold = ck >= 0.8 and len(tokens(kg_text)) >= 4
    return f1, (f1 >= 0.5 or cg >= 0.8 or part_of_gold)


def fingerprint(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()[:12]


def clause_by_paragraph(paragraph_file):
    """Expected clause of every paragraph, from the outline in the paragraph file."""
    starts = {}

    def walk(nodes):
        for node in nodes:
            starts[short(node["paragraphId"])] = node["ref"]
            walk(node.get("children") or [])

    walk(paragraph_file.get("tree") or [])
    current, out = None, {}
    for p in paragraph_file["paragraphs"]:
        current = starts.get(short(p["id"]), current)
        out[short(p["id"])] = current
    return out


REF_NUMBER = re.compile(r"\d+(?:\.\d+)*")


def clause_number(clause):
    if not clause:
        return None
    match = REF_NUMBER.search(clause.get("ref") or "")
    return match.group(0) if match else None


def map_parties(kg, gold):
    def key(name):
        return " ".join(WORD.findall(norm(name)))

    alias_to_gold = {key(n): gp["key"] for gp in gold["parties"] for n in (gp["name"], *gp["aliases"])}
    mapping = {}
    for p in kg.get("parties", []):
        keys = [key(p["name"]), *(key(a) for a in p.get("aliases") or [])]
        mapping[p["id"]] = next((alias_to_gold[k] for k in keys if k in alias_to_gold), None)
    return mapping


def greedy(pairs):
    """One-to-one assignment, best score first. pairs: (score, left, right)."""
    used_l, used_r, out = set(), set(), {}
    for score, left, right in sorted(pairs, key=lambda t: -t[0]):
        if left in used_l or right in used_r:
            continue
        used_l.add(left)
        used_r.add(right)
        out[left] = right
    return out

In [3]:
def evaluate(kg, gold, paragraph_file):
    expected_clause = clause_by_paragraph(paragraph_file)
    party = map_parties(kg, gold)
    clauses = {c["id"]: c for c in kg.get("clauses", [])}
    kg_statements = [dict(n, kind=kind) for coll, kind in STATEMENT_COLLECTIONS.items() for n in kg.get(coll, [])]
    by_id = {s["id"]: s for s in kg_statements}
    gold_statements = {g["id"]: g for g in gold["statements"]}

    def paras(node):
        return {short(p) for p in node.get("paragraphIds") or node.get("paragraphs") or []}

    def burden_ok(g, k):
        return party.get(k.get("burdenPartyId")) in (g["burden"], *g.get("burdenAlso", []))

    def benefit_ok(g, k):
        return party.get(k.get("benefitPartyId")) in (g["benefit"], *g.get("benefitAlso", []))

    # -- statements ------------------------------------------------------- #
    pairs = []
    for g in gold["statements"]:
        for k in kg_statements:
            if not paras(g) & paras(k):
                continue
            f1, ok = similar(g["text"], k.get("text") or k.get("summary") or "")
            if not ok:
                continue
            # Text decides whether two statements are the same provision; once both pass, who
            # it binds decides which statement of the provision it is — so parties outweigh
            # a few more shared words.
            bonus = 0.2 * (k["kind"] == g["kind"]) + 0.3 * burden_ok(g, k) + 0.15 * benefit_ok(g, k)
            # The arbitrators' duties are neutral: they take a KG statement only if no party's does.
            bonus -= 0.5 * g["nonParty"]
            pairs.append((f1 + bonus, g["id"], k["id"]))
    match = greedy(pairs)  # gold id -> kg id
    matched_kg = {v: k for k, v in match.items()}

    def bucket(g):
        if g["nonParty"]:
            return "nonParty"
        return "optional" if g["optional"] else "core"

    found = {b: [] for b in ("core", "optional", "nonParty")}
    total = {b: 0 for b in found}
    for g in gold["statements"]:
        total[bucket(g)] += 1
        if g["id"] in match:
            found[bucket(g)].append(g["id"])

    attrs = {"kind": [], "burden": [], "benefit": [], "clause": []}
    for gid, kid in match.items():
        g, k = gold_statements[gid], by_id[kid]
        # An optional statement is arguable by definition, and so are its parties.
        if bucket(g) != "core":
            continue
        attrs["kind"].append(k["kind"] == g["kind"])
        attrs["burden"].append(burden_ok(g, k))
        attrs["benefit"].append(benefit_ok(g, k))
        attrs["clause"].append(clause_number(clauses.get(k.get("clauseId"))) == expected_clause.get(g["paragraphs"][0]))

    unmatched = {}
    for k in kg_statements:
        if k["id"] in matched_kg:
            continue
        text = k.get("text") or k.get("summary") or ""
        why = "spurious"
        if any(paras(nd) & paras(k) and similar(nd["text"], text)[1] for nd in gold["nonDeontic"]):
            why = "non-deontic"
        else:
            twins = [g for g in gold["statements"] if paras(g) & paras(k) and similar(g["text"], text)[1]]
            if twins and all(g["nonParty"] for g in twins):
                why = "non-party"
            elif twins:
                why = "duplicate"
        unmatched[k["id"]] = why
    neutral = sum(1 for gid in match if gold_statements[gid]["nonParty"]) + sum(w == "non-party" for w in unmatched.values())
    hits = len(match) - sum(1 for gid in match if gold_statements[gid]["nonParty"])
    scored = len(kg_statements) - neutral

    # A bilateral provision is two gold statements with one text; a condition or value
    # pointing at either twin points at the provision.
    def targets(gold_id):
        g = gold_statements[gold_id]
        twins = [t["id"] for t in gold["statements"] if t["text"] == g["text"] and t["paragraphs"] == g["paragraphs"]]
        return {match[t] for t in twins if t in match}

    # -- conditions ------------------------------------------------------- #
    cpairs, clinked = [], set()
    for g in gold["conditions"]:
        for k in kg.get("conditions", []):
            if not paras(g) & paras(k):
                continue
            cg, ck = containment(g["trigger"], k.get("trigger") or "")
            if max(cg, ck) < 0.6:
                continue
            linked = k.get("gatesId") in targets(g["gates"])
            if linked:
                clinked.add((g["id"], k["id"]))
            cpairs.append((max(cg, ck) + linked, g["id"], k["id"]))
    cmatch = greedy(cpairs)
    required_c = [g for g in gold["conditions"] if not g["optional"]]

    # -- values ----------------------------------------------------------- #
    def value_hit(g, k):
        haystack = norm(f"{k.get('amount', '')} {k.get('unit', '')}").replace("$", "").replace(",", "")
        return any(re.search(rf"(?<![\w.]){re.escape(norm(a))}(?![\w])", haystack) for a in g["accept"])

    def value_linked(g, k):
        target = k.get("quantifiesId")
        if g["quantifies"].startswith("clause:"):
            return clause_number(clauses.get(target)) == g["quantifies"].split(":", 1)[1]
        return target in targets(g["quantifies"])

    vpairs, vlinked = [], set()
    for g in gold["values"]:
        for k in kg.get("values", []):
            if paras(g) & paras(k) and value_hit(g, k):
                linked = value_linked(g, k)
                if linked:
                    vlinked.add((g["id"], k["id"]))
                vpairs.append((1 + linked, g["id"], k["id"]))
    vmatch = greedy(vpairs)
    required_v = [g for g in gold["values"] if not g["optional"]]

    def tally(required, found, linked):
        hit = [g["id"] for g in required if g["id"] in found]
        tied = [gid for gid in hit if (gid, found[gid]) in linked]
        return f"{len(hit)}/{len(required)}", f"{len(tied)}/{len(required)}"

    conditions_found, conditions_linked = tally(required_c, cmatch, clinked)
    values_found, values_linked = tally(required_v, vmatch, vlinked)

    def pct(xs):
        return round(100 * sum(xs) / len(xs)) if xs else None

    return {
        "indicators": {
            "statements": len(kg_statements),
            "core_recall_%": round(100 * len(found["core"]) / max(1, total["core"])),
            "precision_%": round(100 * hits / max(1, scored)),
            "optional_found": f"{len(found['optional'])}/{total['optional']}",
            "kind_ok_%": pct(attrs["kind"]),
            "burden_ok_%": pct(attrs["burden"]),
            "benefit_ok_%": pct(attrs["benefit"]),
            "clause_ok_%": pct(attrs["clause"]),
            "non_deontic": sum(w == "non-deontic" for w in unmatched.values()),
            "duplicates": sum(w == "duplicate" for w in unmatched.values()),
            "spurious": sum(w == "spurious" for w in unmatched.values()),
            "conditions_found": conditions_found,
            "conditions_linked": conditions_linked,
            "values_found": values_found,
            "values_linked": values_linked,
            "unknown_parties": sum(1 for v in party.values() if v is None),
        },
        "match": match,
        "unmatched": unmatched,
        "missed": [g["id"] for g in gold["statements"] if bucket(g) == "core" and g["id"] not in match],
        "missed_conditions": [g["id"] for g in required_c if g["id"] not in cmatch],
        "missed_values": [g["id"] for g in required_v if g["id"] not in vmatch],
        "party_map": party,
    }

## The reference holds what it claims

Every span is checked again here, against the paragraph file the graphs are built from: a
reference that drifted from its text would score extractions against words the contract
does not contain.

In [4]:
def verbatim_problems(gold, paragraph_file):
    """Spans of the reference that are not, word for word, in the paragraphs they cite."""
    text = {short(p["id"]): norm(p["text"]) for p in paragraph_file["paragraphs"]}
    spans = [(g["id"], g["paragraphs"], g["text"]) for g in gold["statements"]]
    spans += [(c["id"], c["paragraphs"], c["trigger"]) for c in gold["conditions"]]
    spans += [("nonDeontic", n["paragraphs"], n["text"]) for n in gold["nonDeontic"]]
    return [(sid, pids) for sid, pids, span in spans if norm(span) not in " ".join(text[p] for p in pids)]


for gold_path in sorted(GOLD_DIR.glob("*.json")):
    gold = json.loads(gold_path.read_text(encoding="utf-8"))
    paragraph_file = json.loads((PARAGRAPHS_DIR / gold_path.name).read_text(encoding="utf-8"))
    problems = verbatim_problems(gold, paragraph_file)
    counts = {key: len(gold[key]) for key in ("statements", "nonDeontic", "conditions", "values")}
    print(f"{gold_path.stem[5:60]}: {counts} | spans not verbatim: {problems or 'none'}")

SteelVaultCorp_20081224_10-K_EX-10_16_3074935_EX-10_16_: {'statements': 94, 'nonDeontic': 26, 'conditions': 20, 'values': 13} | spans not verbatim: none


## Runs

`infra/json/kg_runs/<contract>/runs.json` lists every logged extraction with the model,
the pipeline and the prompt that produced it. The graph the app serves is scored too when
it is not one of the logged runs.

Columns: `recall` core statements found · `prec` share of the graph's statements that are
real · `opt` optional statements found · `kind`, `burden`, `benef`,
`clause` share right among the matched ones · `nondeon`, `dup`, `spur` statements with no
counterpart, by reason · `cond`, `val` gold conditions and values found, and `c.link`,
`v.link` those also attached to the right statement · `?party` graph parties that match no
real one.

In [5]:
COLUMNS = {
    "statements": "stmts", "core_recall_%": "recall", "precision_%": "prec", "optional_found": "opt",
    "kind_ok_%": "kind", "burden_ok_%": "burden", "benefit_ok_%": "benef",
    "clause_ok_%": "clause", "non_deontic": "nondeon", "duplicates": "dup", "spurious": "spur",
    "conditions_found": "cond", "conditions_linked": "c.link", "values_found": "val", "values_linked": "v.link",
    "unknown_parties": "?party",
}


def runs_of(stem):
    """Every graph to score for one contract: the logged runs, plus the served graph if it is not one of them."""
    folder = RUNS_DIR / stem
    manifest = json.loads((folder / "runs.json").read_text()) if (folder / "runs.json").exists() else []
    runs = [(row["file"], row, folder / row["file"]) for row in manifest]
    served = KG_DIR / f"{stem}.json"
    if served.exists() and fingerprint(served) not in {fingerprint(path) for _, _, path in runs}:
        runs.append(("served", {"model": "?"}, served))
    return runs


results, metas = {}, {}
for gold_path in sorted(GOLD_DIR.glob("*.json")):
    stem = gold_path.stem
    gold = json.loads(gold_path.read_text(encoding="utf-8"))
    paragraph_file = json.loads((PARAGRAPHS_DIR / f"{stem}.json").read_text(encoding="utf-8"))
    print(f"\n{stem[5:60]}  ({gold['status'].split(' —')[0]})")
    print(f"  {'run':34s}{'model':>14s}" + "".join(f"{label:>8s}" for label in COLUMNS.values()))
    for label, meta, path in runs_of(stem):
        result = evaluate(json.loads(path.read_text(encoding="utf-8")), gold, paragraph_file)
        results[(stem, label)] = result
        metas[(stem, label)] = meta
        row = result["indicators"]
        print(f"  {label[:33]:34s}{str(meta.get('model'))[:13]:>14s}" + "".join(f"{str(row[key]):>8s}" for key in COLUMNS))


SteelVaultCorp_20081224_10-K_EX-10_16_3074935_EX-10_16_  (draft 2.3)
  run                                        model   stmts  recall    prec     opt    kind  burden   benef  clause nondeon     dup    spur    cond  c.link     val  v.link  ?party
  baseline_gpt-4.1.json                    gpt-4.1      83      81      87   13/21     100      98      93      91      10       0       0    8/17    8/17     7/9     6/9       0
  20261005-015915_gpt-6-luna_D2.jso     gpt-6-luna      84      78      72    6/21      96      93      93     100      14       9       0   11/17   11/17     9/9     8/9       0
  20261005-021402_gpt-6-luna_D2.jso     gpt-6-luna      98      93      85   14/21      81      97      97     100       8       6       0   11/17   10/17     9/9     7/9       0
  20261005-021542_gpt-6-luna_D2.jso     gpt-6-luna      84      88      85   10/21      93      97     100      98       7       5       1   10/17   10/17     9/9     7/9       0
  20261005-021727_gpt-6-luna_D2.jso

  20261005-021901_gpt-6-luna_D1.jso     gpt-6-luna      98      87      78   13/21      98     100      95     100      14       6       0   12/17   12/17     4/9     4/9       0


  20261005-022032_gpt-6-luna_D1.jso     gpt-6-luna      89      78      74    8/21      98      98      98      98      14       6       2   11/17   11/17     8/9     6/9       0


  20261005-022155_gpt-6-luna_D1.jso     gpt-6-luna      93      83      77   11/21      95      96      98      98      17       3       0   14/17   14/17     9/9     8/9       0
  20261005-022559_gpt-6.1-sol_D2.js    gpt-6.1-sol     109     100      83   18/21     100     100     100     100      13       3       2   16/17   16/17     9/9     9/9       0
  20261005-022944_gpt-6.1-sol_D2.js    gpt-6.1-sol     101     100      91   19/21     100     100     100     100       6       2       1   16/17   16/17     9/9     9/9       0
  20261005-023331_gpt-6.1-sol_D2.js    gpt-6.1-sol     103     100      89   19/21     100     100     100     100       7       3       1   16/17   16/17     9/9     9/9       0
  20261005-023736_gpt-6.1-sol_D1.js    gpt-6.1-sol     108     100      85   19/21     100     100     100     100      11       3       2   16/17   16/17     9/9     8/9       0


  20261005-024141_gpt-6.1-sol_D1.js    gpt-6.1-sol     104     100      89   20/21     100      97      97     100       7       2       2   16/17   16/17     9/9     8/9       0


  20261005-024611_gpt-6.1-sol_D1.js    gpt-6.1-sol     113     100      81   18/21     100     100     100     100      13       4       4   16/17   16/17     9/9     9/9       0


  20261005-025541_gpt-6.1-sol_D2.js    gpt-6.1-sol      90     100      95   13/21     100     100     100     100       0       3       1   16/17   16/17     9/9     9/9       0
  20261005-025913_gpt-6.1-sol_D2.js    gpt-6.1-sol      90      99      94   13/21     100     100     100     100       1       3       1   16/17   16/17     9/9     9/9       0
  20261005-030342_gpt-6.1-sol_D2.js    gpt-6.1-sol      92     100      94   14/21     100     100     100     100       1       2       2   16/17   16/17     9/9     9/9       0


## By configuration

One extraction of one contract is a draw: the same prompt on the same text does not give
the same graph twice. Runs are grouped by everything that produced them — pipeline, model,
design, prompt fingerprint — and reported as mean and range.

The rule for choosing, fixed before the first comparison was run: a configuration beats
another only if its **worst** run beats the other's **best** on recall and on precision.
Overlapping ranges are a tie, and a tie goes to the cheaper configuration.

In [6]:
from collections import defaultdict
from statistics import mean

SUMMARY = {
    "core_recall_%": "recall", "precision_%": "prec", "kind_ok_%": "kind", "burden_ok_%": "burden",
    "benefit_ok_%": "benef", "clause_ok_%": "clause", "conditions_linked": "c.link", "values_linked": "v.link",
}


def as_number(value):
    """86 -> 86; "11/17" -> 11."""
    return int(value.split("/")[0]) if isinstance(value, str) and "/" in value else value


groups = defaultdict(list)
for (stem, label), result in results.items():
    meta = metas[(stem, label)]
    groups[(stem, meta.get("pipeline"), meta.get("model"), meta.get("design"), meta.get("prompt"))].append((meta, result["indicators"]))

for stem in sorted({key[0] for key in groups}):
    print(f"\n{stem[5:60]} — mean [min-max] over the runs of each configuration")
    print(f"  {'pipeline':12s}{'model':13s}{'design':>7s}{'prompt':>14s}{'n':>3s}{'$/run':>8s}" + "".join(f"{label:>15s}" for label in SUMMARY.values()))
    for key, rows in sorted(groups.items(), key=lambda item: (item[0][0], str(item[0][1]), str(item[0][2]), str(item[0][3]))):
        if key[0] != stem:
            continue
        costs = [m.get("costUsd") for m, _ in rows if m.get("costUsd") is not None]
        cells = []
        for name in SUMMARY:
            values = [as_number(r[name]) for _, r in rows if r[name] is not None]
            cells.append(f"{mean(values):.0f} [{min(values)}-{max(values)}]" if values else "-")
        cost = f"{mean(costs):.3f}" if costs else "-"
        print(f"  {str(key[1])[:11]:12s}{str(key[2])[:12]:13s}{str(key[3]):>7s}{str(key[4])[:12]:>14s}{len(rows):>3d}{cost:>8s}" + "".join(f"{c:>15s}" for c in cells))


SteelVaultCorp_20081224_10-K_EX-10_16_3074935_EX-10_16_ — mean [min-max] over the runs of each configuration
  pipeline    model         design        prompt  n   $/run         recall           prec           kind         burden          benef         clause         c.link         v.link
  chunks-v1   gpt-4.1         None           v_1  1       -     81 [81-81]     87 [87-87]  100 [100-100]     98 [98-98]     93 [93-93]     91 [91-91]        8 [8-8]        6 [6-6]
  skeleton-v2 gpt-6-luna        D1  6844e36950c5  3   0.010     83 [78-87]     76 [74-78]     97 [95-98]    98 [96-100]     97 [95-98]    99 [98-100]     12 [11-14]        6 [4-8]
  skeleton-v2 gpt-6-luna        D2  72d6826f3a3c  1   0.010     78 [78-78]     72 [72-72]     96 [96-96]     93 [93-93]     93 [93-93]  100 [100-100]     11 [11-11]        8 [8-8]
  skeleton-v2 gpt-6-luna        D2  6844e36950c5  3   0.009     91 [88-93]     83 [80-85]     91 [81-98]     97 [97-97]    99 [97-100]    99 [98-100]     10 [10-11]      

### Decision of 2026-10-05 (SteelVault, prompt `6844e36950c5`, reference draft 2.3)

- **Design: D2.** For gpt-6-luna, D2 beats D1 on both measures (worst D2 run above the best
  D1 run); for gpt-6.1-sol they tie and D2 is cheaper. Reading the whole contract does not
  help, and it hurts the small model.
- **Model: gpt-6.1-sol.** Read literally, the rule gives a tie — recall does not overlap
  (100 against 88–93) but precision does (83–91 against 80–85) — and the tie would go to
  gpt-6-luna. The rule was written too strict: a model clearly better on one measure and
  no worse on the other could never win. gpt-6.1-sol was chosen for the recall gap and for
  linking 16 of 17 breach conditions against 10–11, the consequences the exposure analysis
  is built on. It costs about $0.21 a run on this contract against $0.009.

From here on, a configuration wins when it is ahead on one measure without overlap and not
behind on the other; the same rule is read the same way for every later comparison.

### Boilerplate rule (prompt `75f20786f4b8`), kept

The block prompt now tells the model that provisions about how the contract itself works —
governing law, severability, no waiver, entire agreement, deeming, the parties' status —
bind no party. With gpt-6.1-sol on D2, precision went from 88 [83-91] to 94 [94-95], ranges
apart, with recall unchanged (100 [99-100]) and each run cheaper ($0.19 against $0.21), since
there is less to write. What is left: a warranty listed twice, the billing practice of §1
repeated, and the signature-block instruction "Executive two (2) copies...".


## What was missed

Reading the misses keeps the reference honest: a miss that turns out to be arguable is a
reason to mark that gold statement optional, not to celebrate a lower number.

In [7]:
def show(stem, label):
    gold = json.loads((GOLD_DIR / f"{stem}.json").read_text(encoding="utf-8"))
    kg_path = KG_DIR / f"{stem}.json" if label == "served" else RUNS_DIR / stem / label
    kg = json.loads(kg_path.read_text(encoding="utf-8"))
    result = results[(stem, label)]
    gold_by_id = {g["id"]: g for g in gold["statements"]}
    kg_by_id = {n["id"]: n for coll in STATEMENT_COLLECTIONS for n in kg.get(coll, [])}

    print(f"-- gold statements missed ({len(result['missed'])}) --")
    for gid in result["missed"]:
        g = gold_by_id[gid]
        print(f"   {gid:5s} {g['paragraphs'][0]:6s} {g['kind']:11s} {g['text'][:100]}")

    print(f"\n-- graph statements with no gold counterpart ({len(result['unmatched'])}) --")
    for kid, why in result["unmatched"].items():
        node = kg_by_id[kid]
        where = short(node["paragraphIds"][0]) if node.get("paragraphIds") else "-"
        print(f"   {why:12s} {kid:15s} {where:6s} {(node.get('text') or node.get('summary') or '')[:90]}")

    print(f"\n-- conditions missed: {result['missed_conditions']}")
    print(f"-- values missed:     {result['missed_values']}")


first = next(iter(results), None)
if first:
    show(*first)

-- gold statements missed (13) --
   S12   p-13   obligation  Notice will be given to Marketing Affiliate in writing no less than 30 days prior to such increase t
   S16   p-14   obligation  this Agreement will serve as a personal guaranty by the undersigned principal, partner or owner of t
   S39a  p-27   prohibition Such approval shall not be unreasonably withheld
   S39b  p-27   obligation  shall be completed within 48 hours of receipt of Marketing Materials for review
   S41   p-30   right       Equidata may audit, at Equidata’s expense, the Marketing Affiliate’s marketing, practices and activi
   S42   p-30   right       Equidata reserves the right to site inspect Marketing Affiliate’s physical location of business at a
   W1    p-37   obligation  Marketing Affiliate does not engage in any business with respect to, and the Marketing Affiliate Web
   W2    p-38   obligation  The information regarding Marketing Affiliate set forth in this Agreement, and the information provi
   W3  